# NeuroArena: decoder accuracy over the NeuroGrid quality level

This notebook shows the published results of NeuroArena (NG-F17): the tables and both figures of the technical
report (`report/en/report.md`, `report/zh/report.md`). It reads the folder `results/` beside this notebook's
folder, or the folder named by the environment variable `NEUROARENA_RESULTS`, and computes nothing new: every
number comes from the published files, which `python -m neuroarena.report publish RUN_A RUN_B` wrote from two
runs of the same configuration.

**Scope:** the results apply only to the public datasets and decoders studied; no statement about the
performance or safety of a medical device.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

# The published results: NEUROARENA_RESULTS, else results/ beside the notebooks/ folder.
RESULTS = Path(os.environ.get("NEUROARENA_RESULTS") or Path.cwd().parent / "results")
method = json.loads((RESULTS / "method.json").read_text(encoding="utf-8"))
comparison = json.loads((RESULTS / "run_compare.json").read_text(encoding="utf-8"))
curve = pd.read_csv(RESULTS / "curve.csv", dtype={"dataset": str, "decoder": str})
sessions = pd.read_csv(RESULTS / "sessions.csv", dtype={"session": str, "quality_level": "Int64"})
summary = pd.read_csv(RESULTS / "quality_summary.csv")
unmeasured = pd.read_csv(RESULTS / "unmeasured.csv")
runs = " and ".join(method["runs"])
print(f"runs {runs}, mode {method['mode']}, {method['neurogrid']['version']}")

## Method in brief

- **Quality level (A-102).** NeuroGrid checks every recording through its public command line (A-101). For
  six core rules (M06 bad channels, M07 line noise, M08 flat and clipped signal, M09a autoreject, M09b ICLabel)
  NeuroArena divides the value by the effective threshold; the largest ratio gives the level Q1 to Q5, and Q4
  starts at the threshold. A session takes the worst level of its runs; a recording without any measurable
  rule has no level and is counted, never dropped.
- **Degradation (A-103).** One ladder per kind with five levels, applied to the raw signal before filtering:
  line noise, flat channels and artefacts (blinks, muscle bursts, electrode pops). The clean data are level 0.
- **Decoders (A-105).** CSP+LDA as MOABB's pipeline (a filter bank for SSVEP) and EEGNet from braindecode, on
  the CPU with fixed seeds.
- **Evaluation (A-104).** Training and test in the same condition, stratified cross-validation within every
  session; a point of a curve is the mean session accuracy with a bootstrap interval. The two runs were
  compared with `compare_runs` (DoD 1).

The parameters as the runs used them:

In [ ]:
print("level boundaries of r:", method["quality"]["boundaries"])
print("effective thresholds:", method["neurogrid"]["thresholds"])
print("conditions:", ", ".join(method["config"]["conditions"]))
evaluation = method["evaluation"]
print(
    "folds per session:", evaluation["n_folds"], "| bootstrap resamples:", evaluation["bootstrap_resamples"]
)
pd.DataFrame(
    {
        name: {
            "paradigm": spec["paradigm"],
            "subjects": len(method["subjects"][name]),
            "EEG channels": spec["n_eeg_channels"],
            "licence": spec["licence"],
        }
        for name, spec in method["datasets"].items()
    }
).T

## Quality levels of the clean recordings

Recordings per level, the share at Q4 or worse, how often each rule gave the level (the worst rule) and how
often it was not measurable (`quality_summary.csv`, `unmeasured.csv`).

In [ ]:
clean = summary[summary["condition"] == "clean"].set_index("dataset")
level_columns = [c for c in clean.columns if c.startswith("recordings_")] + ["q4_or_worse_share"]
display(clean[["n_recordings", *level_columns]])
display(clean.filter(like="worst_").rename(columns=lambda c: c.removeprefix("worst_")))
display(clean.filter(like="unmeasured_").rename(columns=lambda c: c.removeprefix("unmeasured_")))
unmeasured.groupby(["dataset", "rule_id", "reason"])["recordings"].sum().to_frame()

## Accuracy over the quality level (main figure)

Mean accuracy of the (session, condition) pairs at each session quality level, with the bootstrap interval
and the mean chance level; a level without pairs has no point and no column value.

In [ ]:
display(SVG(filename=str(RESULTS / "accuracy_over_quality.svg")))
quality_axis = curve[curve["axis"] == "quality"]
quality_axis.pivot_table(index=["dataset", "decoder"], columns="x", values=["mean_accuracy", "n_sessions"])

Pairs without a quality level per dataset and decoder (they have no point on the quality axis):

In [ ]:
without = sessions.assign(without_level=sessions["quality_level"].isna())
without.groupby(["dataset", "decoder"])["without_level"].agg(["sum", "count"])

## Accuracy over the degradation level (second figure)

Per dataset and kind, the clean pairs at level 0; under each panel the share of sessions per quality level.

In [ ]:
display(SVG(filename=str(RESULTS / "accuracy_over_degradation.svg")))
degradation_axis = curve[curve["axis"] == "degradation"]
degradation_axis.pivot_table(index=["dataset", "decoder", "kind"], columns="x", values="mean_accuracy")

## Comparison of the two runs (DoD 1)

Equality is promised only per machine and lock (A-106).

In [ ]:
print("equal within the tolerance:", comparison["ok"], "| exactly equal:", comparison["exact"])
print("EEGNet tolerance:", comparison["tolerance"])
pd.Series(comparison["compared"], name="compared")